# Imports

In [ ]:
import tifffile
import matplotlib.pyplot as plt
import os
from scipy import ndimage
import numpy as np
import ast
from itertools import dropwhile
from scipy.optimize import curve_fit
from scipy.signal import butter, lfilter
from skimage.morphology import disk,square
from skimage.morphology import closing
from skimage import filters, data, io, morphology, measure
from skimage import data
import pandas as pd

# Allocating each nucleus a distance from the edge of the colony and pSMAD1/5 intensity value

In [ ]:
psmad=tifffile.imread('')
mask_cp=np.load('', allow_pickle=True).item()

In [ ]:
plt.imshow(psmad) 

In [ ]:
plt.imshow(mask_cp['masks'])

In [ ]:
#extract connected component labelled image from the data

cleaned_image = morphology.remove_small_objects(mask_cp['masks'], min_size=500)

In [ ]:
plt.imshow(cleaned_image)

In [ ]:
#OVERLAPPING NUCLEI MASK AND pSMAD DATA - giving each nucleus an intensity value

average_nucleus_intensity_cp=[]
all_nuclei_cp=np.unique(cleaned_image)
all_nuclei_cp=np.delete(all_nuclei_cp,0)
for i in all_nuclei_cp:
    average_nucleus_intensity_cp.append(np.mean(psmad[cleaned_image==i]))

In [ ]:
#FINDING THE POSITION OF THE CENTRE OF MASS OF EACH NUCLEUS - giving each nucleus a positional value (row,column)

regions_cp = measure.regionprops(cleaned_image)
centre_mass_cp = []

for i,nucleus in enumerate (regions_cp):
    centre_mass_cp.append((int(round(nucleus.centroid[0])),int(round(nucleus.centroid[1]))))

In [ ]:
#CREATE A DILATED AND ERODED MASK

selem=disk(48)
dilated_mask_cp=morphology.binary_closing(cleaned_image, selem)
plt.imshow(dilated_mask_cp)

In [ ]:
background=np.invert(dilated_mask_cp)
plt.imshow(background)
print(background)

In [ ]:
background_signal=np.mean(psmad[background==True])
print(background_signal)

In [ ]:
#NORMALISING ACCORDING TO THE BACKGROUND

average_nucleus_intensity_normalised=[i-background_signal for i in average_nucleus_intensity_cp]

In [ ]:
#NORMALISING BEFORE PLOTTING THE INTENSITIES USING 98th PERCENTILE

percentile_98=np.percentile(average_nucleus_intensity_normalised,98)
print(percentile_98)

average_nucleus_intensity_normalised_perc=[i/(percentile_98) for i in average_nucleus_intensity_normalised]
print(min(average_nucleus_intensity_normalised_perc), max(average_nucleus_intensity_normalised_perc))


In [ ]:
#DISTANCE TRANSFORM CENTRE OF EACH NUCLEUS TO THE COLONY EDGE

distance_transformed_cp=ndimage.morphology.distance_transform_edt(dilated_mask_cp)

distance_cp=[]

for i in centre_mass_cp:
    x=i[0]
    y=i[1]
    distance_cp.append(distance_transformed_cp[x,y])

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(20,5))

rounded_distance=[]
for i in distance_cp:
    rounded_distance.append(int(round(i)))

ax.scatter(rounded_distance,average_nucleus_intensity_normalised_perc)
ax.set_title('normalising by background only')
ax.set_xlabel('distance from edge')
ax.set_ylabel('average pSMAD1/5 signal intensity')

# Writing a csv with averaged intensity for each unique distance  

In [ ]:
from collections import defaultdict

#create a default dictionary 
groups=defaultdict(list)

#itirate through distance and intensity together and the distance as a new key if it doesn't exist and intensity as the value corresponding to that key
#if the key (distance) already exists, adds the new value to that key, creating a list of values for that key

for x,y in zip(rounded_distance,average_nucleus_intensity_normalised_perc):
    groups[x].append(y)
    
unique_distance=[]
mean_intensity=[]

#itirate through keys (x) and corresponding values (y) and create two new lists, one for unique distances and one for average intensity at that distance
for x,y in groups.items():
    unique_distance.append(x)
    mean_intensity.append(np.mean(y))


In [ ]:
plt.scatter(unique_distance, mean_intensity)

In [ ]:
mean_intensity_per_distance_stiff=pd.DataFrame({'Distance':unique_distance,'Intensity':mean_intensity})
mean_per_distance_stiff_csv=''

In [ ]:
mean_intensity_per_distance_stiff.to_csv(mean_per_distance_stiff_csv, mode='a', header=False, index=False)

In [ ]:
mean_intensity_per_distance_soft=pd.DataFrame({'Distance':unique_distance,'Intensity':mean_intensity})
mean_per_distance_soft_csv=''

In [ ]:
mean_intensity_per_distance_soft.to_csv(mean_per_distance_soft, mode='a', header=False, index=False)

# After adding all replicates to the csv, loading and processing the csv as dataframe 

In [ ]:
df_stiff_intensity_per_distance = pd.read_csv('', header=None)

In [ ]:
df_soft_intensity_per_distnace = pd.read_csv('', header=None)

# Binning distances on x axis and averaging the intensity over the whole bin 

In [ ]:
# Define the bins for your x values
bins = np.arange(0, df_stiff_intensity_per_distance[0].max() + 50, 50)

print(df_stiff_intensity_per_distance[0].max())

# Bin the x values using pd.cut
df_stiff_intensity_per_distance['distance_binned'] = pd.cut(df_stiff_intensity_per_distance[0], bins, right=False)

df_stiff_intensity_per_distance = df_stiff_intensity_per_distance[df_stiff_intensity_per_distance[0] <= 810]



# Group by the bins and calculate mean and standard deviation of y for each bin
distances_stiff_binned = df_stiff_intensity_per_distance.groupby('distance_binned')[1]
mean_y_stiff = distances_stiff_binned.mean()
std_y_stiff = distances_stiff_binned.std()

# Get the mid-points of the bins for plotting
bin_centers_stiff = [interval.mid for interval in mean_y_stiff.index]

bin_centers_stiff_micron = []

for i,x in enumerate(bin_centers_stiff):
    bin_centers_stiff_micron.append(x*0.3787879)

# Plotting the mean trend with envelopes
plt.plot(bin_centers_stiff_micron, mean_y_stiff.values, label='Mean y value per bin', color='b')
plt.fill_between(bin_centers_stiff_micron, mean_y_stiff - std_y_stiff, mean_y_stiff + std_y_stiff, color='b', alpha=0.2, label='Envelopes (mean ± std)')
plt.xlabel('x (binned)')
plt.ylabel('Mean y')
plt.legend()
plt.show()

In [ ]:
# Define the bins for your x values
bins = np.arange(0, df_soft_intensity_per_distnace[0].max() + 50, 50)

# Bin the x values 
df_soft_intensity_per_distnace['distance_binned'] = pd.cut(df_soft_intensity_per_distnace[0], bins, right=False)

print(df_soft_intensity_per_distnace[0].max())

df_soft_intensity_per_distnace = df_soft_intensity_per_distnace[df_soft_intensity_per_distnace[0] <= 800]

# Group by the bins and calculate mean and standard deviation of y for each bin
distances_soft_binned = df_soft_intensity_per_distnace.groupby('distance_binned')[1]
mean_y_soft = distances_soft_binned.mean()
std_y_soft = distances_soft_binned.std()

# Get the mid-points of the bins for plotting
bin_centers_soft = [interval.mid for interval in mean_y_soft.index]

bin_centers_soft_micron = []

for i,x in enumerate(bin_centers_soft):
    bin_centers_soft_micron.append(x*0.3787879)

# Plotting the mean trend with envelopes
plt.plot(bin_centers_soft_micron, mean_y_soft.values, label='Mean y value per bin', color='b')
plt.fill_between(bin_centers_soft_micron, mean_y_soft - std_y_soft, mean_y_soft + std_y_soft, color='b', alpha=0.2, label='Envelopes (mean ± std)')
plt.xlabel('x (binned)')
plt.ylabel('Mean y')
plt.legend()
plt.show()

# Save the final csvs with all data that will be plotted

In [ ]:
df_stiff_intensity_per_distnace.to_csv('',header=False, index=False)

In [ ]:
df_soft_intensity_per_distance.to_csv('',header=False, index=False)

# Final plot

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(3,4))

ax.plot(bin_centers_stiff_micron, mean_y_stiff.values, label='stiff', color='#63B8FF')
ax.fill_between(bin_centers_stiff_micron, mean_y_stiff - std_y_stiff, mean_y_stiff + std_y_stiff, color='#63B8FF', alpha=0.3)

ax.plot(bin_centers_soft_micron, mean_y_soft.values, label='stiff + laminin', color='#76448C')
ax.fill_between(bin_centers_soft_micron, mean_y_soft - std_y_soft, mean_y_soft + std_y_soft, color='#76448C', alpha=0.3)

ax.set_xlabel('distance from colony edge ($\mathrm{\mu}$m)')
ax.set_ylabel('pSMAD1/5 intensity')
ax.legend()
plt.tight_layout()

plt.rcParams['pdf.fonttype'] = 42